In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [9]:
from langchain_openai import ChatOpenAI

PROVIDER = "groq"

LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini", 
    },
}
MAX_RETRIES = 5  


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed, 
        max_retries=MAX_RETRIES,
    )

llm = get_llm(temperature=0.7) 

In [10]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()

format_instructions = parser.get_format_instructions()
print(format_instructions)

print(parser.parse("명동 칼국수, 부산 돼지국밥, 전주 비빔밥"))

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
['명동 칼국수', '부산 돼지국밥', '전주 비빔밥']


In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 한국 여행과 문화를 잘 아는 추천 전문가입니다.\n"
     "규칙:\n"
     "- 사용자가 말한 분야와 관련된 **한국의** 유명한 장소나 활동을 정확히 5개 추천하세요.\n"
     "- 각 항목은 '지역 + 이름' 형태의 짧은 명사구로 쓰세요. (예: 전주 비빔밥)\n"
     "- 각 항목 안에는 쉼표를 절대 쓰지 마세요.\n"
     "- 번호, 설명, 따옴표 없이 항목만 출력하세요.\n\n"
     "{format_instructions}"),
    ("user", "분야: {category}"),
]).partial(format_instructions=format_instructions)

llm = get_llm(temperature=0.5)

chain = prompt | llm | parser

In [12]:
category = input("관심 분야를 입력하세요 (예: 음식, 스포츠, 영화): ").strip()

result = chain.invoke({"category": category})

print(f"입력 분야: {category}")
print(f"결과 타입: {type(result).__name__}")
print(result)

if len(result) != 5:
    print(f" 항목 수가 {len(result)}개입니다. 프롬프트 규칙을 확인하세요.")

입력 분야: 스포츠
결과 타입: list
['서울 올림픽공원', '부산 해운대 수영장', '제주 한라산 트레일', '강릉 동해 서핑', '대구 수성구 축구장']


In [13]:
categories = ["음식", "스포츠", "영화", "자연"]

results = chain.batch([{"category": c} for c in categories])

for c, items in zip(categories, results):
    status = " "if len(items) == 5 else f"{len(items)}개"
    print(f"[{c}] {status}")
    for i, item in enumerate(items, 1):
        print(f"   {i}. {item}")

[음식]  
   1. 전주 비빔밥
   2. 부산 밀면
   3. 제주 흑돼지
   4. 서울 김치찌개
   5. 경주 한과
[스포츠]  
   1. 강원 평창 휘닉스파크
   2. 부산 해운대 서핑
   3. 서울 올림픽공원
   4. 인천 송도 월드컵경기장
   5. 제주 올레 트레킹
[영화]  
   1. 부산 국제영화제
   2. 제주 섭지코지
   3. 경주 석굴암
   4. 서울 남산타워
   5. 전주 한옥마을
[자연]  
   1. 제주 한라산
   2. 강원 설악산
   3. 경북 울릉도
   4. 전라 지리산
   5. 충남 태안해안
